In [1]:
!pip install -q torch torchvision transformers peft bitsandbytes accelerate nltk rouge-score evaluate bert_score


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import zipfile
from huggingface_hub import hf_hub_download

REPO_ID = "uitnlp/OpenViVQA-dataset"
BASE_DIR = "data/openvivqa"
os.makedirs(BASE_DIR, exist_ok=True)

def download_and_extract(zip_filename, json_filename, split_name):
    print(f"--- Đang xử lý tập {split_name.upper()} ---")
    json_path = hf_hub_download(repo_id=REPO_ID, filename=json_filename, repo_type="dataset", local_dir=BASE_DIR)
    
    img_dir = os.path.join(BASE_DIR, split_name)
    if not os.path.exists(img_dir):
        os.makedirs(img_dir, exist_ok=True)
        print(f"Đang tải {zip_filename} (~vài GB, vui lòng đợi)...")
        zip_path = hf_hub_download(repo_id=REPO_ID, filename=zip_filename, repo_type="dataset", local_dir=BASE_DIR)
        print(f"Đang giải nén {zip_filename}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(img_dir)
        print("Giải nén xong!")
    else:
        print(f"Thư mục ảnh {img_dir} đã tồn tại, bỏ qua tải/giải nén.")
        
    return json_path, img_dir

def parse_openvivqa_json(json_path, img_dir):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    images_dict = data.get("images", {})
    annotations_dict = data.get("annotations", {})
    
    # Tạo mapping name -> absolute_path để tìm ảnh nhanh
    img_name_to_path = {}
    for root, _, files in os.walk(img_dir):
        for file in files:
            img_name_to_path[file] = os.path.join(root, file)
            
    parsed_data = []
    missing = 0
    for ann_id, ann_info in annotations_dict.items():
        img_id_str = str(ann_info["image_id"])
        img_filename = images_dict.get(img_id_str)
        
        if img_filename and img_filename in img_name_to_path:
            parsed_data.append({
                "image": img_name_to_path[img_filename],
                "question": ann_info["question"],
                "answer": ann_info["answer"]
            })
        else:
            missing += 1
            
    print(f"Parse xong {len(parsed_data)} cặp QA. (Thiếu ảnh: {missing})")
    return parsed_data

# Quá trình này sẽ tốn khoảng 5-10 phút để tải file ZIP nếu là lần đầu.
train_json, train_dir = download_and_extract("train-images.zip", "vlsp2023_train_data.json", "train")
val_json, val_dir = download_and_extract("dev-images.zip", "vlsp2023_dev_data.json", "val")
test_json, test_dir = download_and_extract("test-images.zip", "vlsp2023_test_data.json", "test")

train_data = parse_openvivqa_json(train_json, train_dir)
val_data = parse_openvivqa_json(val_json, val_dir)
test_data = parse_openvivqa_json(test_json, test_dir)

print(f"\nTổng: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

c:\Users\NGUYEN DONG QUAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Đang xử lý tập TRAIN ---
Thư mục ảnh data/openvivqa\train đã tồn tại, bỏ qua tải/giải nén.
--- Đang xử lý tập VAL ---
Thư mục ảnh data/openvivqa\val đã tồn tại, bỏ qua tải/giải nén.
--- Đang xử lý tập TEST ---
Thư mục ảnh data/openvivqa\test đã tồn tại, bỏ qua tải/giải nén.
Parse xong 30833 cặp QA. (Thiếu ảnh: 0)
Parse xong 3545 cặp QA. (Thiếu ảnh: 0)
Parse xong 14035 cặp QA. (Thiếu ảnh: 0)

Tổng: Train=30833, Val=3545, Test=14035


In [3]:
import os, json, random, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
from transformers import AutoModel, AutoTokenizer, AutoProcessor, Qwen2VLForConditionalGeneration, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import evaluate
import numpy as np
import matplotlib.pyplot as plt

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Sử dụng thiết bị:', device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

[nltk_data] Downloading package wordnet to C:\Users\NGUYEN DONG
[nltk_data]     QUAN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\NGUYEN DONG
[nltk_data]     QUAN\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Sử dụng thiết bị: cuda


In [4]:
class AnswerVocab:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.idx = 4

    def add_sentence(self, sentence):
        for word in sentence.lower().split():
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def __len__(self): return len(self.word2idx)

    def encode(self, sentence, max_len=12):
        tokens = [1] # <SOS>
        for word in sentence.lower().split():
            tokens.append(self.word2idx.get(word, 3))
        tokens.append(2) # <EOS>
        # Pad
        if len(tokens) < max_len:
            tokens.extend([0] * (max_len - len(tokens)))
        return tokens[:max_len]

    def decode(self, token_ids):
        words = []
        for tid in token_ids:
            if tid == 2: break # EOS
            if tid > 3: words.append(self.idx2word[tid])
        return " ".join(words)

vocab = AnswerVocab()
print("Đang xây dựng từ điển Câu trả lời (Vocab) từ tập Train...")
for item in train_data:
    vocab.add_sentence(item['answer'])
print(f"Kích thước từ điển (Vocab size): {len(vocab)}")

# Lưu Vocab ra file để Gradio App.py có thể đọc lại
import json
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab.word2idx, f, ensure_ascii=False)
print("Đã lưu vocab.json cho ứng dụng Web.")

phobert_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class VQADataset(Dataset):
    def __init__(self, data_list, vocab, tokenizer, transform=None):
        self.data = data_list
        self.transform = transform
        self.vocab = vocab
        self.tokenizer = tokenizer

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        try:
            image = Image.open(item["image"]).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))

        if self.transform: image = self.transform(image)

        q_encoded = self.tokenizer(item["question"], padding='max_length', max_length=20, truncation=True, return_tensors="pt")
        a_encoded = torch.tensor(self.vocab.encode(item["answer"]))

        return {
            "image": image,
            "input_ids": q_encoded.input_ids.squeeze(0),
            "attention_mask": q_encoded.attention_mask.squeeze(0),
            "target_ids": a_encoded,
            "raw_answer": item["answer"]
        }

def get_dataloaders(batch_size=32):
    train_transform = T.Compose([
        T.RandomResizedCrop(224), T.RandomHorizontalFlip(), T.RandomRotation(15),
        T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_test_transform = T.Compose([
        T.Resize((224, 224)), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = VQADataset(train_data, vocab, phobert_tokenizer, train_transform)
    val_dataset = VQADataset(val_data, vocab, phobert_tokenizer, val_test_transform)
    test_dataset = VQADataset(test_data, vocab, phobert_tokenizer, val_test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders()
print("Số batch Train/Val/Test:", len(train_loader), len(val_loader), len(test_loader))

Đang xây dựng từ điển Câu trả lời (Vocab) từ tập Train...
Kích thước từ điển (Vocab size): 6421
Đã lưu vocab.json cho ứng dụng Web.
Số batch Train/Val/Test: 964 111 439


In [5]:
import warnings
warnings.filterwarnings("ignore")  # tắt warning 

class VQAEvaluator:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(
            ['rougeL'],
            use_stemmer=False
        )

        try:
            self.bertscore = evaluate.load("bertscore")
            print("BERTScore loaded OK")
        except Exception as e:
            self.bertscore = None
            print(f"BERTScore không load được: {e}")

    def evaluate_batch(self, predictions, ground_truths_list):
        # ép chắc chắn thành list
        predictions = list(predictions)
        ground_truths_list = list(ground_truths_list)

        results = {
            "accuracy": [],
            "bleu": [],
            "rougeL": [],
            "meteor": []
        }

        smoothie = SmoothingFunction().method4

        # Accuracy / BLEU / ROUGE-L / METEOR
        for pred, gts in zip(predictions, ground_truths_list):
            pred_clean = pred.strip().lower()

            if isinstance(gts, str):
                gts = [gts]

            # Accuracy
            acc = 1.0 if any(
                pred_clean == gt.strip().lower()
                for gt in gts
            ) else 0.0
            results["accuracy"].append(acc)

            pred_tokens = pred_clean.split()
            refs = [
                gt.strip().lower().split()
                for gt in gts
            ]

            results["bleu"].append(
                sentence_bleu(
                    refs,
                    pred_tokens,
                    smoothing_function=smoothie
                )
            )

            rouge_scores = [
                self.rouge.score(gt, pred)["rougeL"].fmeasure
                for gt in gts
            ]

            results["rougeL"].append(
                max(rouge_scores) if rouge_scores else 0.0
            )

            try:
                results["meteor"].append(
                    meteor_score(refs, pred_tokens)
                )
            except:
                results["meteor"].append(0.0)

        avg_results = {
            k: float(np.mean(v))
            for k, v in results.items()
        }

        # BERTScore
        if self.bertscore is not None and len(predictions) > 0:
            flat_refs = [
                gts[0] if isinstance(gts, list) else gts
                for gts in ground_truths_list
            ]

            try:
                bs = self.bertscore.compute(
                    predictions=predictions,
                    references=flat_refs,
                    lang="vi",
                    model_type="bert-base-multilingual-cased"
                )

                avg_results["bertscore_f1"] = float(
                    np.mean(bs["f1"])
                )

            except Exception as e:
                print(f"BERTScore lỗi: {e}")
                avg_results["bertscore_f1"] = 0.0
        else:
            avg_results["bertscore_f1"] = 0.0

        return avg_results

evaluator = VQAEvaluator()

BERTScore loaded OK


In [6]:
# ENCODERS (dùng chung cho cả A1 và A2)
class ImageEncoder(nn.Module):
    """
    ResNet50 giữ lại spatial features [B, 49, embed_size]
    thay vì ép về 1 vector — cho phép decoder attend vào từng vùng ảnh.
    """
    def __init__(self, embed_size=512, finetune_layers=1):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Bỏ avgpool + fc → giữ feature map 7×7
        backbone = list(resnet.children())[:-2]

        # Freeze tất cả, chỉ unfreeze N layer cuối
        self.frozen   = nn.Sequential(*backbone[:-finetune_layers])
        self.finetune = nn.Sequential(*backbone[-finetune_layers:])
        for p in self.frozen.parameters():
            p.requires_grad = False

        self.proj     = nn.Linear(2048, embed_size)
        self.norm     = nn.LayerNorm(embed_size)

    def forward(self, images):
        with torch.no_grad():
            x = self.frozen(images)          # [B, C, H, W] — frozen
        x = self.finetune(x)                 # gradient chảy qua đây
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H * W, C)   # [B, 49, 2048]
        return self.norm(self.proj(x))       # [B, 49, embed_size]

class TextEncoder(nn.Module):
    """
    PhoBERT giữ toàn bộ sequence [B, L, embed_size]
    thay vì chỉ lấy [CLS] — câu hỏi dài hơn được biểu diễn đầy đủ hơn.
    """
    def __init__(self, embed_size=512, finetune_layers=2):
        super().__init__()
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")

        # Freeze tất cả, chỉ unfreeze N transformer layer cuối
        for p in self.phobert.parameters():
            p.requires_grad = False
        for layer in self.phobert.encoder.layer[-finetune_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.proj = nn.Linear(768, embed_size)
        self.norm = nn.LayerNorm(embed_size)

    def forward(self, input_ids, attention_mask):
        out = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        return self.norm(self.proj(out.last_hidden_state))  # [B, L, embed_size]

# FUSION — dùng chung, concat img + txt sequences
class MultimodalFusion(nn.Module):
    """
    Concat img_seq và txt_seq theo chiều sequence,
    sau đó project về hidden_size để decoder sử dụng làm memory.
    """
    def __init__(self, embed_size=512, hidden_size=512):
        super().__init__()
        self.proj = nn.Linear(embed_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, img_feat, txt_feat):
        # img_feat: [B, 49, embed_size]
        # txt_feat: [B, L,  embed_size]
        memory = torch.cat([img_feat, txt_feat], dim=1)   # [B, 49+L, embed_size]
        return self.norm(self.proj(memory))               # [B, 49+L, hidden_size]

# MODEL A1 — LSTM Decoder
class VQAModelA1(nn.Module):
    """
    Encoder → Fusion → LSTM Decoder
    Memory được nén về 1 vector qua attention pooling
    trước khi đưa vào h0 của LSTM — phù hợp với kiến trúc RNN.
    """
    def __init__(self, vocab_size, embed_size=512, hidden_size=512,
                 finetune_img=1, finetune_txt=2, dropout=0.3):
        super().__init__()
        self.img_encoder = ImageEncoder(embed_size, finetune_layers=finetune_img)
        self.txt_encoder = TextEncoder(embed_size,  finetune_layers=finetune_txt)
        self.fusion      = MultimodalFusion(embed_size, hidden_size)

        # Attention pooling: nén sequence memory → 1 vector cho h0 LSTM
        self.attn_pool   = nn.Linear(hidden_size, 1)

        self.embed       = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.dropout     = nn.Dropout(dropout)
        self.lstm        = nn.LSTM(embed_size, hidden_size,
                                   num_layers=2, batch_first=True,
                                   dropout=dropout)
        self.fc_out      = nn.Linear(hidden_size, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)

    def _encode(self, images, input_ids, attention_mask):
        img_feat = self.img_encoder(images)
        txt_feat = self.txt_encoder(input_ids, attention_mask)
        memory   = self.fusion(img_feat, txt_feat)         # [B, S, hidden]

        # Attention pooling → h0: [B, hidden]
        scores   = self.attn_pool(memory).squeeze(-1)      # [B, S]
        weights  = torch.softmax(scores, dim=-1).unsqueeze(-1)
        pooled   = (memory * weights).sum(dim=1)           # [B, hidden]
        return pooled

    def forward(self, images, input_ids, attention_mask,
                answer_seq=None, max_len=30, eos_id=2):
        pooled = self._encode(images, input_ids, attention_mask)

        # h0, c0 cho 2-layer LSTM: [num_layers, B, hidden]
        h0 = pooled.unsqueeze(0).repeat(2, 1, 1)
        c0 = torch.zeros_like(h0)

        if answer_seq is not None:
            # Teacher forcing
            x = self.dropout(self.embed(answer_seq))       # [B, T, embed]
            out, _ = self.lstm(x, (h0, c0))
            return self.fc_out(self.dropout(out))          # [B, T, vocab_size]

        else:
            # Autoregressive greedy decoding
            B = images.size(0)
            curr  = torch.full((B, 1), 1, dtype=torch.long, device=images.device)
            h, c  = h0, c0
            preds = []

            for _ in range(max_len):
                emb        = self.dropout(self.embed(curr))
                out, (h, c) = self.lstm(emb, (h, c))
                next_tok   = self.fc_out(out).argmax(dim=-1)   # [B, 1]
                preds.append(next_tok)
                curr = next_tok
                if (next_tok == eos_id).all():
                    break

            return torch.cat(preds, dim=1)                 # [B, T_gen]

# MODEL A2 — Transformer Decoder
class VQAModelA2(nn.Module):
    """
    Encoder → Fusion → Transformer Decoder
    Memory giữ nguyên dạng sequence — decoder cross-attend
    vào toàn bộ img+txt tokens, tận dụng đầy đủ attention.
    """
    def __init__(self, vocab_size, embed_size=512, hidden_size=512,
                 num_heads=8, num_layers=4, dropout=0.2,
                 finetune_img=1, finetune_txt=2):
        super().__init__()
        assert hidden_size % num_heads == 0, \
            f"hidden_size={hidden_size} phải chia hết cho num_heads={num_heads}"

        self.img_encoder = ImageEncoder(embed_size, finetune_layers=finetune_img)
        self.txt_encoder = TextEncoder(embed_size,  finetune_layers=finetune_txt)
        self.fusion      = MultimodalFusion(embed_size, hidden_size)

        self.embed       = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.pos_embed   = nn.Embedding(512, hidden_size)
        self.dropout     = nn.Dropout(dropout)

        decoder_layer    = nn.TransformerDecoderLayer(
            d_model=hidden_size, nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout, batch_first=True,
            norm_first=True          # Pre-LN: ổn định hơn khi train
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.out_norm    = nn.LayerNorm(hidden_size)
        self.fc_out      = nn.Linear(hidden_size, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)
        nn.init.normal_(self.embed.weight, mean=0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0, std=0.02)

    def _encode(self, images, input_ids, attention_mask):
        img_feat = self.img_encoder(images)
        txt_feat = self.txt_encoder(input_ids, attention_mask)
        return self.fusion(img_feat, txt_feat)             # [B, 49+L, hidden]

    def _decode_step(self, tgt_seq, memory):
        """1 bước decode — dùng cho cả train (full seq) và inference (từng token)."""
        T   = tgt_seq.size(1)
        pos = torch.arange(T, device=tgt_seq.device).unsqueeze(0)
        tgt = self.dropout(self.embed(tgt_seq) + self.pos_embed(pos))

        mask = nn.Transformer.generate_square_subsequent_mask(
            T, device=tgt_seq.device
        )
        pad_mask = (tgt_seq == 0)                          # [B, T] — ignore padding

        out = self.decoder(tgt, memory,
                           tgt_mask=mask,
                           tgt_key_padding_mask=pad_mask)
        return self.fc_out(self.out_norm(out))             # [B, T, vocab_size]

    def forward(self, images, input_ids, attention_mask,
                answer_seq=None, max_len=30, eos_id=2):
        memory = self._encode(images, input_ids, attention_mask)

        if answer_seq is not None:
            # Teacher forcing: input [:-1], label [1:] — xử lý ở training loop
            return self._decode_step(answer_seq, memory)  # [B, T, vocab_size]

        else:
            # Autoregressive greedy decoding
            B    = images.size(0)
            seq  = torch.full((B, 1), 1, dtype=torch.long, device=images.device)

            for _ in range(max_len):
                logits    = self._decode_step(seq, memory)
                next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # [B, 1]
                seq       = torch.cat([seq, next_tok], dim=1)
                if (next_tok == eos_id).all():
                    break

            return seq[:, 1:]                              # bỏ <SOS>

model_a1 = VQAModelA1(
    vocab_size=len(vocab),
    embed_size=512, hidden_size=512,
    finetune_img=1, finetune_txt=2,
    dropout=0.3
).to(device)

model_a2 = VQAModelA2(
    vocab_size=len(vocab),
    embed_size=512, hidden_size=512,
    num_heads=8, num_layers=4,
    finetune_img=1, finetune_txt=2,
    dropout=0.1
).to(device)

# Kiểm tra nhanh số params trainable
def count_params(model, name):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{name}] Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M '
          f'({100*trainable/total:.1f}%)')

count_params(model_a1, 'A1')
count_params(model_a2, 'A2')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7627.34it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14934.17it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                        

[A1] Total: 171.0M | Trainable: 41.6M (24.3%)
[A2] Total: 183.9M | Trainable: 54.5M (29.6%)


In [7]:
import os, json
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

CKPT_DIR = 'A1_A2_model'
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, name, is_best=False):
    state = {
        'epoch':           epoch,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'train_loss':      train_loss,
        'val_loss':        val_loss,
    }
    path = os.path.join(CKPT_DIR, f'ckpt_{name}_epoch{epoch}.pth')
    torch.save(state, path)

    if is_best:
        best_path = os.path.join(CKPT_DIR, f'best_{name}.pth')
        torch.save(state, best_path)
        print(f'  [Best saved] val_loss={val_loss:.4f} → {best_path}')
    print(f'  [Checkpoint] {path}')

def load_checkpoint(model, optimizer, scheduler, name):
    files = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith(f'ckpt_{name}_epoch')])
    if not files:
        print(f'  Không có checkpoint cho {name}, bắt đầu từ đầu.')
        return 0, [], []

    latest = os.path.join(CKPT_DIR, files[-1])
    ckpt   = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    if 'scheduler_state' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state'])

    # Khôi phục loss history từ các file checkpoint đã lưu
    train_hist, val_hist = [], []
    for f in files:
        c = torch.load(os.path.join(CKPT_DIR, f), map_location='cpu', weights_only=False)
        train_hist.append(c['train_loss'])
        val_hist.append(c.get('val_loss', float('nan')))

    print(f'  [Resumed] {files[-1]} (epoch {ckpt["epoch"]})')
    return ckpt['epoch'] + 1, train_hist, val_hist

def run_epoch(model, loader, criterion, optimizer=None, is_train=True):
    """Dùng chung cho cả train và validation."""
    model.train() if is_train else model.eval()
    total_loss, n_batches = 0.0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            images         = batch['image'].to(device)
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            target_ids     = batch['target_ids'].to(device)

            outputs = model(images, input_ids, attention_mask,
                            answer_seq=target_ids[:, :-1])           # input: [:-1]
            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                target_ids[:, 1:].reshape(-1)                        # label: [1:]
            )

            if is_train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # FIX: clip gradient
                optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

    return total_loss / max(n_batches, 1)

def train_model(model, train_loader, val_loader, name, total_epochs=5, lr=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    # Resume nếu có checkpoint — khôi phục cả loss history
    start_epoch, train_hist, val_hist = load_checkpoint(model, optimizer, scheduler, name)

    if start_epoch >= total_epochs:
        print(f'{name} đã train đủ {total_epochs} epochs, bỏ qua.')
        return train_hist, val_hist

    best_val_loss = min(val_hist, default=float('inf'))
    print(f'Bắt đầu huấn luyện {name} từ epoch {start_epoch + 1}/{total_epochs}...')

    for epoch in range(start_epoch, total_epochs):
        train_loss = run_epoch(model, train_loader, criterion, optimizer, is_train=True)
        val_loss   = run_epoch(model, val_loader,   criterion, optimizer=None, is_train=False)

        train_hist.append(train_loss)
        val_hist.append(val_loss)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        print(f'[{name}] Epoch {epoch+1}/{total_epochs} '
              f'— Train: {train_loss:.4f} | Val: {val_loss:.4f}'
              f'{" ← best" if is_best else ""}')

        scheduler.step(val_loss)
        save_checkpoint(model, optimizer, scheduler,
                        epoch, train_loss, val_loss, name, is_best=is_best)

    # Plot cả train và val loss
    epochs = range(1, len(train_hist) + 1)
    plt.figure()
    plt.plot(epochs, train_hist, marker='o', label='Train loss')
    plt.plot(epochs, val_hist,   marker='s', label='Val loss')
    plt.title(f'Loss — {name}')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.tight_layout(); plt.show()

    return train_hist, val_hist

# RUN
train_model(model_a1, train_loader, val_loader, name='a1', total_epochs=10)
train_model(model_a2, train_loader, val_loader, name='a2', total_epochs=10)

  [Resumed] ckpt_a1_epoch9.pth (epoch 9)
a1 đã train đủ 10 epochs, bỏ qua.
  [Resumed] ckpt_a2_epoch9.pth (epoch 9)
a2 đã train đủ 10 epochs, bỏ qua.


([4.261408816482022,
  2.81889495377224,
  2.2875515900212204,
  1.9592173746265316,
  1.7094885379199665,
  1.508235463401094,
  1.3423932807450472,
  1.1918319719708312,
  1.0588156314435342,
  0.8443284837907775],
 [3.4532269121290327,
  2.8687144528638133,
  2.611182645634488,
  2.49371561381194,
  2.4168540507823497,
  2.379094967970977,
  2.3843864696519868,
  2.4084067580936193,
  2.4093883365124196,
  2.3997121012962617])

In [8]:
SAVE_DIR = "./saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Đang lưu mô hình A1 và A2...")
torch.save(
    model_a1.module.state_dict()
    if isinstance(model_a1, nn.DataParallel)
    else model_a1.state_dict(),
    f"{SAVE_DIR}/model_a1.pth"
)

torch.save(
    model_a2.module.state_dict()
    if isinstance(model_a2, nn.DataParallel)
    else model_a2.state_dict(),
    f"{SAVE_DIR}/model_a2.pth"
)

print("Mô hình A1, A2 đã được lưu tại thư mục:", SAVE_DIR)

Đang lưu mô hình A1 và A2...
Mô hình A1, A2 đã được lưu tại thư mục: ./saved_models
